# Imports

In [1]:
# Cell 1: Environment check & imports
import os
from pathlib import Path
import numpy as np
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm

# optional imports used later
import scipy.linalg
from scipy.interpolate import Rbf

# Print versions
print("numpy", np.__version__)
print("PIL", Image.__version__)
print("cv2", cv2.__version__)

numpy 2.2.6
PIL 10.2.0
cv2 4.12.0


# Path Setup

In [2]:
# === CELL 2: Path Setup ===

from pathlib import Path

# Adjust these paths to match your project structure
PROJECT_ROOT = Path("..").resolve()       # notebook is inside /notebooks/
DATA_DIR = PROJECT_ROOT / "data"

DATA_RAW       = DATA_DIR / "raw" / "OriginalImages"
DATA_LANDMARKS = DATA_DIR / "landmarks" 
DATA_PROCESSED = DATA_DIR / "processed"
DATA_ALIGNED   = DATA_PROCESSED / "aligned"

# Create directories if missing
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_LANDMARKS.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_ALIGNED.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:   ", PROJECT_ROOT)
print("DATA_RAW:       ", DATA_RAW)
print("DATA_LANDMARKS: ", DATA_LANDMARKS)
print("DATA_PROCESSED: ", DATA_PROCESSED)
print("DATA_ALIGNED:   ", DATA_ALIGNED)
print("\nPath setup complete ✔")

PROJECT_ROOT:    D:\Projects 2025\Caricature Generation
DATA_RAW:        D:\Projects 2025\Caricature Generation\data\raw\OriginalImages
DATA_LANDMARKS:  D:\Projects 2025\Caricature Generation\data\landmarks
DATA_PROCESSED:  D:\Projects 2025\Caricature Generation\data\processed
DATA_ALIGNED:    D:\Projects 2025\Caricature Generation\data\processed\aligned

Path setup complete ✔


# Landmark Loading Helper

In [3]:
# Cell 3: landmark loader + visualization helper
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import cv2

def load_landmarks(p):
    p = Path(p)
    if not p.exists(): return None
    try:
        arr = np.load(str(p))
        if arr is None or arr.size == 0:
            return None
        arr = np.asarray(arr)
        if arr.ndim == 2 and arr.shape[1] == 2:
            return arr.astype(np.float32)
    except Exception:
        return None
    return None

def show_with_landmarks(img_path, lm_path=None, figsize=(4,4)):
    img = np.array(Image.open(img_path).convert('RGB'))
    plt.figure(figsize=figsize)
    plt.imshow(img); plt.axis('off')
    if lm_path and Path(lm_path).exists():
        pts = load_landmarks(lm_path)
        if pts is not None:
            for (x,y) in pts:
                cv2.circle(img, (int(round(x)),int(round(y))), 2, (255,0,0), -1)
            plt.figure(figsize=figsize)
            plt.imshow(img); plt.axis('off')
    return

# Landmark Extraction (AFTER resize & pad)

In [4]:
# Cell 4: Landmark extraction using face_alignment (DEMO friendly)
# Install: pip install face-alignment torch  (run in your env if missing)

import os
from pathlib import Path
import numpy as np
from PIL import Image
from tqdm import tqdm

# Demo config
DEMO_LIMIT = 50   # set number of images to run now; set to None to run all
DEVICE = 'cuda' if (lambda: __import__('torch').cuda.is_available())() else 'cpu'

# instantiate face_alignment
import face_alignment
from face_alignment import LandmarksType
fa = face_alignment.FaceAlignment(LandmarksType.TWO_D, device=DEVICE, flip_input=False)

LANDROOT = Path(DATA_LANDMARKS)
PREVIEW_DIR = Path(DATA_PROCESSED) / "previews"
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
FAIL_LOG = Path(DATA_PROCESSED) / "extract_failures.txt"

# collect images
# collect images (photos only: filename stem starts with 'p')
all_files = []
for root, _, files in os.walk(DATA_RAW):
    for fn in files:
        # skip non-image extensions quickly
        if not fn.lower().endswith(('.jpg','.jpeg','.png','.bmp')):
            continue
        # check filename stem (without extension) starts with 'p' -> photo
        stem = Path(fn).stem
        if not stem.lower().startswith('p'):
            continue
        all_files.append(os.path.join(root, fn))
        if DEMO_LIMIT is not None and len(all_files) >= DEMO_LIMIT:
            break
    if DEMO_LIMIT is not None and len(all_files) >= DEMO_LIMIT:
        break

print("Images to process (demo):", len(all_files))

count_ok = 0
count_fail = 0

def largest_face_index(preds):
    areas = []
    for pts in preds:
        xs = pts[:,0]; ys = pts[:,1]
        areas.append((xs.max()-xs.min())*(ys.max()-ys.min()))
    return int(np.argmax(areas))

for img_path in tqdm(all_files):
    rel = os.path.relpath(img_path, DATA_RAW)
    base = os.path.splitext(rel)[0]
    out_base = str(LANDROOT / base)

    if Path(out_base + ".npy").exists():
        continue

    try:
        img = np.array(Image.open(img_path).convert('RGB'))
    except Exception as e:
        with open(FAIL_LOG, "a") as f: f.write(f"{out_base}\timage_load_failed:{e}\n")
        count_fail += 1
        continue

    try:
        preds = fa.get_landmarks(img)
    except Exception as e:
        with open(FAIL_LOG, "a") as f: f.write(f"{out_base}\tfa_exception:{e}\n")
        count_fail += 1
        continue

    if preds is None:
        with open(FAIL_LOG, "a") as f: f.write(f"{out_base}\tno_faces_detected\n")
        count_fail += 1
        continue

    idx = largest_face_index(preds)
    pts = preds[idx].astype(np.float32)
    if pts.size == 0 or pts.shape[1] != 2:
        with open(FAIL_LOG, "a") as f: f.write(f"{out_base}\tinvalid_landmark_shape\n")
        count_fail += 1
        continue

    try:
        Path(out_base).parent.mkdir(parents=True, exist_ok=True)
        np.save(out_base + ".npy", pts.astype(np.float32))
        # save preview overlay
        vis = img.copy()
        for (x,y) in pts:
            cv2.circle(vis, (int(round(x)), int(round(y))), 2, (255,0,0), -1)
        preview_path = PREVIEW_DIR / (Path(out_base).name + "_preview.png")
        Image.fromarray(vis).save(preview_path)
        count_ok += 1
    except Exception as e:
        with open(FAIL_LOG, "a") as f: f.write(f"{out_base}\tsave_failed:{e}\n")
        count_fail += 1
        continue

print(f"Landmark extraction done. OK: {count_ok}, Failed: {count_fail}")
print("Failure log:", FAIL_LOG)
print("Previews saved to:", PREVIEW_DIR)

Images to process (demo): 50


100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [06:06<00:00,  7.32s/it]

Landmark extraction done. OK: 50, Failed: 0
Failure log: D:\Projects 2025\Caricature Generation\data\processed\extract_failures.txt
Previews saved to: D:\Projects 2025\Caricature Generation\data\processed\previews


# Compute Mean Normalized Shape (for alignment)

In [5]:
# Cell 5: Build mean normalized shape (run after landmarks exist)
import os
import numpy as np
from pathlib import Path

MIN_LANDMARKS = 5
OUT_MEAN_PATH = Path(DATA_PROCESSED) / "aligned" / "mean_norm_shape.npy"
OUT_MEAN_PATH.parent.mkdir(parents=True, exist_ok=True)

def bbox_diag(pts):
    xs, ys = pts[:,0], pts[:,1]
    return np.linalg.norm([xs.max()-xs.min(), ys.max()-ys.min()])

def normalize_shape(pts):
    centroid = pts.mean(axis=0)
    scale = bbox_diag(pts)
    if scale < 1e-6:
        scale = 1.0
    norm = (pts - centroid) / scale
    return norm

def build_mean_normalized_shape(landmark_root=DATA_LANDMARKS):
    all_norms = []
    counts = {}
    for root, _, files in os.walk(landmark_root):
        for fn in files:
            if not fn.lower().endswith(".npy"):
                continue
            # skip landmark files not belonging to photos (filenames start with 'p')
            stem = Path(fn).stem
            if not stem.lower().startswith('p'):
                continue
            path = Path(root) / fn
            try:
                pts = np.load(str(path))
            except:
                continue
            if pts is None or pts.size == 0 or pts.shape[0] < MIN_LANDMARKS:
                continue
            all_norms.append(normalize_shape(pts))
            counts[pts.shape[0]] = counts.get(pts.shape[0], 0) + 1

    if len(all_norms) == 0:
        raise RuntimeError("No valid landmarks found in DATA_LANDMARKS to build mean shape.")

    common_n = max(counts, key=counts.get)
    final_norms = [n for n in all_norms if n.shape[0] == common_n]
    print("Using landmark count:", common_n, "shapes used:", len(final_norms))
    mean_norm_shape = np.mean(np.stack(final_norms, axis=0), axis=0)
    np.save(OUT_MEAN_PATH, mean_norm_shape)
    print("Saved mean normalized shape to:", OUT_MEAN_PATH)
    return mean_norm_shape

mean_norm = build_mean_normalized_shape()

Using landmark count: 68 shapes used: 50
Saved mean normalized shape to: D:\Projects 2025\Caricature Generation\data\processed\aligned\mean_norm_shape.npy


# Build Canonical Target Shape + Similarity Transform Helper

In [6]:
# Cell 6: Build canonical target_pts and transform helper
import numpy as np
import cv2
from pathlib import Path

OUTPUT_SIZE = (256,256)
TARGET_FACE_SCALE = 0.55
MEAN_NORM_PATH = Path(DATA_PROCESSED) / "aligned" / "mean_norm_shape.npy"
TARGET_PTS_PATH = Path(DATA_PROCESSED) / "aligned" / "target_pts.npy"
TARGET_PTS_PATH.parent.mkdir(parents=True, exist_ok=True)

if MEAN_NORM_PATH.exists():
    mean_norm = np.load(str(MEAN_NORM_PATH))
else:
    try:
        mean_norm
    except NameError:
        raise RuntimeError("Mean normalized shape not found. Run Cell 5 first.")
    else:
        mean_norm = np.asarray(mean_norm)

w_out, h_out = OUTPUT_SIZE
center_out = np.array([w_out/2.0, h_out/2.0], dtype=np.float32)
target_scale = min(w_out, h_out) * TARGET_FACE_SCALE
target_pts = (mean_norm * target_scale + center_out).astype(np.float32)
np.save(str(TARGET_PTS_PATH), target_pts)
print("Saved target_pts:", TARGET_PTS_PATH)

def estimate_similarity_transform(src_pts, dst_pts, method='ransac'):
    src = np.asarray(src_pts, dtype=np.float32)
    dst = np.asarray(dst_pts, dtype=np.float32)
    if src.shape[0] < 3:
        raise ValueError("At least 3 points required")
    if method == 'ransac':
        M, inliers = cv2.estimateAffinePartial2D(src, dst, method=cv2.RANSAC,
                                                 ransacReprojThreshold=3.0, maxIters=2000)
        if M is None:
            M, inliers = cv2.estimateAffinePartial2D(src, dst, method=cv2.LMEDS)
    else:
        M, inliers = cv2.estimateAffinePartial2D(src, dst, method=cv2.LMEDS)
    if M is None:
        raise RuntimeError("estimateAffinePartial2D failed")
    return M

Saved target_pts: D:\Projects 2025\Caricature Generation\data\processed\aligned\target_pts.npy


# Batch Alignment: Warp All Images Into Canonical Space

In [7]:
# Cell 7: Batch alignment (no-crop). Warps full image into canonical canvas and saves both image+aligned landmarks.
import os
import cv2
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm

OUTPUT_SIZE = (256,256)
w_out, h_out = OUTPUT_SIZE
DEMO_LIMIT = None   # set to small integer for testing; None for all
VERBOSE = True
BG_FILL = (30,30,30)

RAWROOT = Path(DATA_RAW)
LANDROOT = Path(DATA_LANDMARKS)
ALIGNED_ROOT = Path(DATA_ALIGNED)
ALIGNED_ROOT.mkdir(parents=True, exist_ok=True)
FAIL_LOG = ALIGNED_ROOT / "alignment_failures_no_crop.txt"

def transform_landmarks(pts, M):
    ones = np.ones((pts.shape[0],1), dtype=np.float32)
    homo = np.concatenate([pts.astype(np.float32), ones], axis=1)
    M_full = np.vstack([M, [0,0,1]])
    return (M_full @ homo.T).T[:, :2]

# gather
# gather (photos only)
raw_images = []
for root, _, files in os.walk(RAWROOT):
    for fn in files:
        if not fn.lower().endswith(('.jpg','.jpeg','.png','.bmp')):
            continue
        stem = Path(fn).stem
        if not stem.lower().startswith('p'):
            continue
        raw_images.append(os.path.join(root, fn))


count_ok=0; count_fail=0; count_skipped=0
if FAIL_LOG.exists(): FAIL_LOG.unlink()

for img_path in tqdm(raw_images):
    if DEMO_LIMIT is not None and count_ok >= DEMO_LIMIT:
        break

    rel = os.path.relpath(img_path, RAWROOT)
    stem = os.path.splitext(rel)[0]
    lm_src = LANDROOT / (stem + ".npy")
    out_img_path = ALIGNED_ROOT / (stem + "_aligned.png")
    out_lm_path  = ALIGNED_ROOT / (stem + "_aligned.npy")

    if not lm_src.exists():
        with open(FAIL_LOG, "a") as f: f.write(f"{rel}\tno_landmarks\n")
        count_fail += 1
        if VERBOSE: print("Skipping (no landmarks):", rel)
        continue

    if out_img_path.exists() and out_lm_path.exists():
        count_skipped += 1
        continue

    try:
        src_pts = np.load(str(lm_src)).astype(np.float32)
        img = np.array(Image.open(img_path).convert('RGB'))

        target_path = Path(DATA_PROCESSED) / "aligned" / "target_pts.npy"
        if target_path.exists():
            dst_pts = np.load(str(target_path)).astype(np.float32)
            if src_pts.shape[0] != dst_pts.shape[0]:
                nmin = min(src_pts.shape[0], dst_pts.shape[0])
                src_use = src_pts[:nmin]; dst_use = dst_pts[:nmin]
            else:
                src_use = src_pts; dst_use = dst_pts
            M, inliers = cv2.estimateAffinePartial2D(src_use.astype(np.float32), dst_use.astype(np.float32),
                                                     method=cv2.RANSAC, ransacReprojThreshold=3.0, maxIters=2000)
            if M is None:
                M, inliers = cv2.estimateAffinePartial2D(src_use.astype(np.float32), dst_use.astype(np.float32),
                                                         method=cv2.LMEDS)
        else:
            centroid = src_pts.mean(axis=0)
            scale = max(1.0, np.linalg.norm([src_pts[:,0].max()-src_pts[:,0].min(), src_pts[:,1].max()-src_pts[:,1].min()]))
            s = (min(w_out, h_out) * 0.55) / scale
            tx = w_out/2.0 - s*centroid[0]
            ty = h_out/2.0 - s*centroid[1]
            M = np.array([[s,0,tx],[0,s,ty]], dtype=np.float32)

        if M is None:
            with open(FAIL_LOG, "a") as f: f.write(f"{rel}\testimation_failed\n")
            count_fail += 1
            continue

        # warp full image into canonical canvas; keep entire canvas (no crop)
        warped = cv2.warpAffine(img, M, (w_out, h_out), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=BG_FILL)
        transformed_pts = transform_landmarks(src_pts, M)

        out_img_path.parent.mkdir(parents=True, exist_ok=True)
        Image.fromarray(warped).save(str(out_img_path))
        np.save(str(out_lm_path), transformed_pts.astype(np.float32))

        count_ok += 1
    except Exception as e:
        with open(FAIL_LOG, "a") as f: f.write(f"{rel}\texception:{e}\n")
        count_fail += 1
        if VERBOSE: print("Exception:", rel, e)
        continue

print("Done. aligned:", count_ok, "skipped:", count_skipped, "fail:", count_fail)
print("Aligned files:", ALIGNED_ROOT)

 10%|███████▊                                                                     | 608/5974 [00:01<00:04, 1184.07it/s]

Skipping (no landmarks): Alan Rickman\P00005.jpg
Skipping (no landmarks): Alan Rickman\P00006.jpg
Skipping (no landmarks): Alan Rickman\P00007.jpg
Skipping (no landmarks): Alan Rickman\P00008.jpg
Skipping (no landmarks): Alan Rickman\P00009.jpg
Skipping (no landmarks): Alan Rickman\P00010.jpg
Skipping (no landmarks): Alan Rickman\P00011.jpg
Skipping (no landmarks): Alan Rickman\P00012.jpg
Skipping (no landmarks): Alan Rickman\P00013.jpg
Skipping (no landmarks): Alan Rickman\P00014.jpg
Skipping (no landmarks): Alan Rickman\P00015.jpg
Skipping (no landmarks): Alan Rickman\P00016.jpg
Skipping (no landmarks): Alan Rickman\P00017.jpg
Skipping (no landmarks): Alan Rickman\P00018.jpg
Skipping (no landmarks): Alan Rickman\P00019.jpg
Skipping (no landmarks): Alan Rickman\P00020.jpg
Skipping (no landmarks): Alan Rickman\P00021.jpg
Skipping (no landmarks): Alan Rickman\P00022.jpg
Skipping (no landmarks): Alan Rickman\P00023.jpg
Skipping (no landmarks): Alan Rickman\P00024.jpg
Skipping (no landmar

 19%|██████████████▊                                                             | 1162/5974 [00:02<00:02, 1916.93it/s]

Skipping (no landmarks): Bill Clinton\P00002.jpg
Skipping (no landmarks): Bill Clinton\P00003.jpg
Skipping (no landmarks): Bill Clinton\P00004.jpg
Skipping (no landmarks): Bill Clinton\P00005.jpg
Skipping (no landmarks): Bill Clinton\P00006.jpg
Skipping (no landmarks): Bill Clinton\P00007.jpg
Skipping (no landmarks): Bill Clinton\P00008.jpg
Skipping (no landmarks): Bill Clinton\P00009.jpg
Skipping (no landmarks): Bill Clinton\P00010.jpg
Skipping (no landmarks): Bill Clinton\P00011.jpg
Skipping (no landmarks): Bill Clinton\P00012.jpg
Skipping (no landmarks): Bill Clinton\P00013.jpg
Skipping (no landmarks): Bill Clinton\P00014.jpg
Skipping (no landmarks): Bill Clinton\P00015.jpg
Skipping (no landmarks): Bill Clinton\P00016.jpg
Skipping (no landmarks): Bill Clinton\P00017.jpg
Skipping (no landmarks): Bill Clinton\P00018.jpg
Skipping (no landmarks): Bill Clinton\P00019.jpg
Skipping (no landmarks): Bill Clinton\P00020.jpg
Skipping (no landmarks): Bill Murray\P00001.jpg
Skipping (no landmark

 29%|██████████████████████▏                                                     | 1748/5974 [00:02<00:01, 2398.79it/s]

...

Skipping (no landmarks): Penelope Cruz\P00019.jpg
Skipping (no landmarks): Penelope Cruz\P00020.jpg
Skipping (no landmarks): Penelope Cruz\P00021.jpg
Skipping (no landmarks): Penelope Cruz\P00022.jpg
Skipping (no landmarks): Penelope Cruz\P00023.jpg
Skipping (no landmarks): Penelope Cruz\P00024.jpg
Skipping (no landmarks): Penelope Cruz\P00025.jpg
Skipping (no landmarks): Penelope Cruz\P00026.jpg
Skipping (no landmarks): Penelope Cruz\P00027.jpg
Skipping (no landmarks): Penelope Cruz\P00028.jpg
Skipping (no landmarks): Peter Falk\P00001.jpg
Skipping (no landmarks): Peter Falk\P00002.jpg
Skipping (no landmarks): Peter Falk\P00003.jpg
Skipping (no landmarks): Peter Falk\P00004.jpg
Skipping (no landmarks): Peter Falk\P00005.jpg
Skipping (no landmarks): Peter Falk\P00006.jpg
Skipping (no landmarks): Peter Falk\P00007.jpg
Skipping (no landmarks): Peter Falk\P00008.jpg
Skipping (no landmarks): Peter Falk\P00009.jpg
Skipping (no landmarks): Peter Falk\P00010.jpg
Skipping (no landmarks): Peter

 93%|██████████████████████████████████████████████████████████████████████▍     | 5540/5974 [00:03<00:00, 2704.19it/s]

Skipping (no landmarks): Sean Connery\P00001.jpg
Skipping (no landmarks): Sean Connery\P00002.jpg
Skipping (no landmarks): Sean Connery\P00003.jpg
Skipping (no landmarks): Sean Connery\P00004.jpg
Skipping (no landmarks): Sean Connery\P00005.jpg
Skipping (no landmarks): Sean Connery\P00006.jpg
Skipping (no landmarks): Sean Connery\P00007.jpg
Skipping (no landmarks): Sean Connery\P00008.jpg
Skipping (no landmarks): Sean Connery\P00009.jpg
Skipping (no landmarks): Sean Connery\P00010.jpg
Skipping (no landmarks): Sean Connery\P00011.jpg
Skipping (no landmarks): Sean Connery\P00012.jpg
Skipping (no landmarks): Sean Connery\P00013.jpg
Skipping (no landmarks): Sean Connery\P00014.jpg
Skipping (no landmarks): Sean Connery\P00015.jpg
Skipping (no landmarks): Sean Connery\P00016.jpg
Skipping (no landmarks): Sean Connery\P00017.jpg
Skipping (no landmarks): Sean Connery\P00018.jpg
Skipping (no landmarks): Sean Connery\P00019.jpg
Skipping (no landmarks): Sean Connery\P00020.jpg
Skipping (no landmar

100%|████████████████████████████████████████████████████████████████████████████| 5974/5974 [00:04<00:00, 1449.97it/s]

Skipping (no landmarks): Tracy Morgan\P00016.jpg
Skipping (no landmarks): Tracy Morgan\P00017.jpg
Skipping (no landmarks): Tracy Morgan\P00018.jpg
Skipping (no landmarks): Tracy Morgan\P00019.jpg
Skipping (no landmarks): Tracy Morgan\P00020.jpg
Skipping (no landmarks): Victoria Beckham\P00001.jpg
Skipping (no landmarks): Victoria Beckham\P00002.jpg
Skipping (no landmarks): Victoria Beckham\P00003.jpg
Skipping (no landmarks): Victoria Beckham\P00004.jpg
Skipping (no landmarks): Victoria Beckham\P00005.jpg
Skipping (no landmarks): Victoria Beckham\P00006.jpg
Skipping (no landmarks): Victoria Beckham\P00007.jpg
Skipping (no landmarks): Victoria Beckham\P00008.jpg
Skipping (no landmarks): Victoria Beckham\P00009.jpg
Skipping (no landmarks): Victoria Beckham\P00010.jpg
Skipping (no landmarks): Victoria Beckham\P00011.jpg
Skipping (no landmarks): Victoria Beckham\P00012.jpg
Skipping (no landmarks): Victoria Beckham\P00013.jpg
Skipping (no landmarks): Victoria Beckham\P00014.jpg
Skipping (no 

# TPS Warp (Exaggeration) on Aligned Images

In [8]:
# Cell 8: Full-image TPS warp that preserves background (inverse TPS + gaussian falloff)
import numpy as np
import cv2
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import scipy.linalg

# Parameters
REG = 1e-3
GRID_STEP = 2
FALLOFF_RADIUS_FACTOR = 1.8
FALLOFF_SHARPNESS = 2.0
DEMO_N = None      # None -> all aligned images; or set to small int for demo
OUT_SUBDIR = "tps_full_preserved"
DEBUG = False

ALIGNED_ROOT = Path(DATA_ALIGNED)
OUT_ROOT = Path(DATA_PROCESSED) / "aligned" / OUT_SUBDIR
OUT_ROOT.mkdir(parents=True, exist_ok=True)

def _tps_kernel(r):
    out = np.zeros_like(r, dtype=np.float64)
    nz = r > 0
    out[nz] = (r[nz]**2) * np.log(r[nz])
    return out

def solve_tps(ctrl_pts, target_vals, reg=1e-3):
    N = ctrl_pts.shape[0]
    diff = ctrl_pts[:, None, :] - ctrl_pts[None, :, :]
    r = np.sqrt((diff**2).sum(axis=2))
    K = _tps_kernel(r)
    P = np.concatenate([np.ones((N,1)), ctrl_pts], axis=1)
    A = np.zeros((N+3, N+3), dtype=np.float64)
    A[:N, :N] = K + reg * np.eye(N)
    A[:N, N:] = P
    A[N:, :N] = P.T
    rhs = np.zeros((N+3,), dtype=np.float64)
    rhs[:N] = target_vals.astype(np.float64)
    coeffs = scipy.linalg.solve(A, rhs, assume_a='sym')
    w = coeffs[:N]; a = coeffs[N:]
    return w, a

def tps_eval_grid(ctrl_pts, w, a, xs, ys):
    XX, YY = np.meshgrid(xs, ys)
    pts = np.stack([XX.ravel(), YY.ravel()], axis=1)
    diff = pts[:, None, :] - ctrl_pts[None, :, :]
    r = np.sqrt((diff**2).sum(axis=2))
    U = _tps_kernel(r)
    fflat = U.dot(w) + a[0] + a[1]*pts[:,0] + a[2]*pts[:,1]
    return fflat.reshape(len(ys), len(xs))

# default exaggeration
def default_exaggeration(pts):
    pts2 = pts.copy()
    n = pts2.shape[0]
    if n >= 17:
        jaw_idx = list(range(0,17))
    else:
        jaw_idx = list(range(n))
    center = pts2.mean(axis=0)
    for i in jaw_idx:
        pts2[i] = center + 1.20 * (pts2[i] - center)
    return pts2

aligned_imgs = sorted(list(ALIGNED_ROOT.rglob("*_aligned.png")))
# filter to photo stems only (e.g., P0001_aligned.png)
aligned_imgs = [p for p in aligned_imgs if p.stem.lower().startswith('p')]

if len(aligned_imgs) == 0:
    raise RuntimeError("No aligned images found. Run alignment first.")

to_process = aligned_imgs if DEMO_N is None else aligned_imgs[:DEMO_N]

for p in to_process:
    lm_p = Path(str(p).replace("_aligned.png", "_aligned.npy"))
    if not lm_p.exists():
        print("Skip (no lm):", p); continue
    out_path = OUT_ROOT / (p.stem + "_tps_full.png")

    try:
        img = np.array(Image.open(p).convert("RGB"))
        h, w = img.shape[:2]
        pts = np.load(str(lm_p)).astype(np.float64)

        # compute dst (exaggerated) in image coords
        dst_pts = default_exaggeration(pts.copy())

        # inverse TPS: ctrl = dst, values = src coords
        w_x, a_x = solve_tps(dst_pts, pts[:,0], reg=REG)
        w_y, a_y = solve_tps(dst_pts, pts[:,1], reg=REG)

        # evaluate on coarse grid then upsample
        if GRID_STEP <= 1:
            xs = np.arange(0, w); ys = np.arange(0, h)
            map_x = tps_eval_grid(dst_pts, w_x, a_x, xs, ys)
            map_y = tps_eval_grid(dst_pts, w_y, a_y, xs, ys)
        else:
            xs_c = np.arange(0, w, GRID_STEP); ys_c = np.arange(0, h, GRID_STEP)
            map_xc = tps_eval_grid(dst_pts, w_x, a_x, xs_c, ys_c)
            map_yc = tps_eval_grid(dst_pts, w_y, a_y, xs_c, ys_c)
            map_x = cv2.resize(map_xc.astype(np.float32), (w, h), interpolation=cv2.INTER_CUBIC)
            map_y = cv2.resize(map_yc.astype(np.float32), (w, h), interpolation=cv2.INTER_CUBIC)

        # identity grid
        id_x = np.tile(np.arange(w, dtype=np.float32)[None, :], (h, 1))
        id_y = np.tile(np.arange(h, dtype=np.float32)[:, None], (1, w))

        # falloff
        centroid = pts.mean(axis=0)
        bbox_diag = np.linalg.norm([pts[:,0].max()-pts[:,0].min(), pts[:,1].max()-pts[:,1].min()])
        radius = bbox_diag * FALLOFF_RADIUS_FACTOR if bbox_diag>1e-6 else max(h,w)*0.25
        xx, yy = np.meshgrid(np.arange(w), np.arange(h))
        dist = np.sqrt((xx - centroid[0])**2 + (yy - centroid[1])**2)
        weight = np.exp(- (dist / (radius + 1e-8))**2 * FALLOFF_SHARPNESS).astype(np.float32)

        final_map_x = (1.0 - weight) * id_x + weight * map_x.astype(np.float32)
        final_map_y = (1.0 - weight) * id_y + weight * map_y.astype(np.float32)

        final_map_x = np.clip(final_map_x, 0.0, float(w-1)).astype(np.float32)
        final_map_y = np.clip(final_map_y, 0.0, float(h-1)).astype(np.float32)

        src_u8 = np.clip(img, 0, 255).astype(np.uint8)
        warped = cv2.remap(src_u8, final_map_x, final_map_y, interpolation=cv2.INTER_CUBIC,
                           borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0))

        out_path.parent.mkdir(parents=True, exist_ok=True)
        Image.fromarray(warped).save(str(out_path))

        if DEBUG:
            fig, ax = plt.subplots(1,3, figsize=(12,4))
            ax[0].imshow(img); ax[0].set_title("src"); ax[0].axis('off')
            ax[1].imshow(warped); ax[1].set_title("warped"); ax[1].axis('off')
            ax[2].imshow(weight, cmap='inferno'); ax[2].set_title("weight"); ax[2].axis('off')
            plt.show()

    except Exception as e:
        print("Failed:", p, e)
        continue

print("TPS full-image outputs saved to:", OUT_ROOT)

TPS full-image outputs saved to: D:\Projects 2025\Caricature Generation\data\processed\aligned\tps_full_preserved
